# Tomography Globe — 3D Printable Hollow Hemispheres

This notebook produces two OBJ hemisphere files suitable for full-colour 3D printing.
It uses ETOPO topography for surface displacement and a seismic tomography depth
slice for vertex colouring.

The workflow uses `create_hollow_hemispheres`, which:
1. Splits the displaced outer shell at the equator (with a capped plane cut).
2. Boolean-subtracts the smooth inner sphere from each half.
3. Returns two **watertight, manifold** hollow hemispheres.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    PolygonDisplacer,
    GridColourer,
    calculate_displacement_scale
)

## 1. Model Parameters

In [ ]:
# --- Globe geometry ---
outer_points = 1000000   # number of vertices in the outer shell
inner_points = 20000     # number of vertices in the inner shell
model_radius_mm = 40.0   # 80 mm diameter globe
inner_scale = 0.8        # inner void radius as a fraction of the outer

# --- Displacement ---
vert_exagg = 50          # vertical exaggeration factor for topography
earth_radius_km = 6371.0
tomography_displacement_scale = -1.5
displace_inner_with_tomo = True
coastline_step_mm = 0.5

# --- Boolean engine ---
# 'manifold' (recommended, fast) or 'blender' (requires Blender installed)
boolean_engine = 'manifold'

## 2. Generate Base Spheres

In [ ]:
print("Generating outer sphere...")
model = GlobeModel.from_fibonacci(outer_points, model_radius_mm)
print(f"  Outer: {len(model.outer_vertices):,} vertices, {len(model.outer_faces):,} faces")

print("Generating inner sphere...")
from globe3d.mesh import generate_sphere_points_fibonacci
inner_v, inner_f = generate_sphere_points_fibonacci(inner_points, model_radius_mm * inner_scale)
model.inner_vertices = inner_v
model.inner_faces = inner_f
print(f"  Inner: {len(model.inner_vertices):,} vertices, {len(model.inner_faces):,} faces")

## 3. Load Geographic Grids

In [ ]:
dem_grid = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"
colour_grid = "../inputs/s40_depth_slice_2850.grd"

print("Loading ETOPO topography grid...")
topo_grid_data = GeographicGrid.from_netcdf(dem_grid, lat_var='lat', lon_var='lon', data_var='z')
print(f"  Shape: {topo_grid_data.grid.shape}")
print(f"  Latitude range: {topo_grid_data.lats.min()} - {topo_grid_data.lats.max()}")
print(f"  Longitude range: {topo_grid_data.lons.min()} - {topo_grid_data.lons.max()}")
print(f"  Z range: {topo_grid_data.grid.min()} - {topo_grid_data.grid.max()}")

print("Loading tomography grid...")
tomo_grid_data = GeographicGrid.from_netcdf(colour_grid, lat_var='y', lon_var='x', data_var='z')
print(f"  Shape: {tomo_grid_data.grid.shape}")
print(f"  Latitude range: {tomo_grid_data.lats.min()} - {tomo_grid_data.lats.max()}")
print(f"  Longitude range: {tomo_grid_data.lons.min()} - {tomo_grid_data.lons.max()}")
print(f"  Z range: {np.nanmin(tomo_grid_data.grid)} - {np.nanmax(tomo_grid_data.grid)}")

## 4. Displace Vertices

Offset the vertices radially according to the grids and geographic context provided (in this example, a surface topography model from ETOPO, and a slice from the S40RTS tomography model, as per above)

Retrieve the coastline shapefiles from:
[Natural Earth Downloads](https://www.naturalearthdata.com/downloads/110m-physical-vectors/)
(e.g., download `ne_110m_land.zip` and extract to `../inputs/coastlines/`).


In [ ]:
coastline_shp = "../inputs/coastlines/ne_110m_land.shp"

scale = calculate_displacement_scale(model_radius_mm, earth_radius_km, vertical_exagg=vert_exagg)
print(f"Displacement scale factor: {scale:.6e}")

print("Displacing outer vertices with topography...")
model.displace(GridDisplacer(topo_grid_data, show_progress=True), scale=scale)

print("Displacing outer vertices with tomography...")
model.displace(GridDisplacer(tomo_grid_data, show_progress=True), scale=tomography_displacement_scale)

if displace_inner_with_tomo:
    print("Displacing interior vertices with tomography...")
    tomo_displacer = GridDisplacer(tomo_grid_data, show_progress=True)
    model.inner_vertices = tomo_displacer(model.inner_vertices, scale=tomography_displacement_scale * inner_scale)

print("Applying 1mm step at the coastlines using shapefile...")
model.displace(PolygonDisplacer(coastline_shp, displacement=coastline_step_mm))

## 5. Split & Hollow into Hemispheres

`create_hollow_hemispheres` performs the correct order of operations:
1. Splits the displaced outer globe at the equator with a capped plane cut.
2. Boolean-subtracts the smooth inner sphere from each half.

The `manifold` boolean engine creates its own triangulation for the annular
cap (the flat ring between the outer and inner shells), so no additional
cap refinement is needed.

The result is two **watertight, manifold** hollow hemispheres — ready for slicing.

In [ ]:
print("Splitting and hollowing hemispheres...")
print(f"  Boolean engine: {boolean_engine}")

thickness_mm = model_radius_mm * (1.0 - inner_scale)
top_half, bottom_half = model.generate_hemispheres(
    hollow=True,
    thickness=thickness_mm,
    engine=boolean_engine
)

if top_half is not None and bottom_half is not None:
    print(f"  Top:    {len(top_half.vertices):,} verts, {len(top_half.faces):,} faces, "
          f"watertight={top_half.is_watertight}, volume={top_half.volume:.1f} mm³")
    print(f"  Bottom: {len(bottom_half.vertices):,} verts, {len(bottom_half.faces):,} faces, "
          f"watertight={bottom_half.is_watertight}, volume={bottom_half.volume:.1f} mm³")
else:
    print("ERROR: Hemisphere creation failed.")

## 6. Colour Options

In [ ]:
# --- Colouring Options ---
# Option 1: Colour according to boundaries
cmap_bounds = mcolors.ListedColormap(['red', 'white', 'blue'])
norm = mcolors.BoundaryNorm([-10, -0.5, 0.5, 10], cmap_bounds.N)
cmap = cmap_bounds
colour_kwargs = {'norm': norm}

# Option 2: Colour using a continuous matplotlib cmap
# cmap = 'RdBu'
# colour_kwargs = {'vmin': -1, 'vmax': 1}

# Option 3: Colour using a discretised version of a matplotlib cmap
# cmap = plt.get_cmap('RdBu', 7)
# colour_kwargs = {'vmin': -2, 'vmax': 2}

## 7. Preview Colour Map

In [ ]:
plt.figure(figsize=(10, 5))
lon_grid, lat_grid = np.meshgrid(tomo_grid_data.lons, tomo_grid_data.lats)
plt.pcolormesh(lon_grid, lat_grid, tomo_grid_data.grid, cmap=cmap, **colour_kwargs, shading='auto')
plt.colorbar(label='Value')
plt.title('2D Preview of Colour Grid')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 8. Assign Colours & Export

In [ ]:
if top_half is not None and bottom_half is not None:
    print("Assigning colors to split halves using recipe...")
    tomo_colourer = GridColourer(tomo_grid_data, colormap=cmap, **colour_kwargs)
    model.colour(tomo_colourer, selection='outward_facing')
    
    # Regenerate hemispheres with coloring applied
    top_half, bottom_half = model.generate_hemispheres(
        hollow=True,
        thickness=thickness_mm,
        engine=boolean_engine
    )
    
    top_colors = top_half.visual.vertex_colors[:, :3].astype(float) / 255.0
    bottom_colors = bottom_half.visual.vertex_colors[:, :3].astype(float) / 255.0
    
    import os
    os.makedirs('../outputs', exist_ok=True)
    print("Exporting...")
    from globe3d.io import write_obj_with_vertex_colors
    write_obj_with_vertex_colors('../outputs/tomo_globe_top.obj', top_half.vertices, top_half.faces, top_colors)
    write_obj_with_vertex_colors('../outputs/tomo_globe_bottom.obj', bottom_half.vertices, bottom_half.faces, bottom_colors)
    print("Exported split globes successfully.")
else:
    print("Skipping export — hemisphere creation failed.")

## 9. Visualise final globe in 3D

In [ ]:
import trimesh
from IPython.display import display

if top_half is not None and bottom_half is not None:
    top_mesh = top_half
    bottom_mesh = bottom_half

    print('Upper mesh: (click and drag to rotate, mouse-scroll to zoom)')
    display(top_mesh.show())
    print('Lower mesh: (click and drag to rotate, mouse-scroll to zoom)')
    display(bottom_mesh.show())